## Attempt: Surface-Extracted Eigenvalue Problem — Not Working

**Status:** ❌ Does not produce consistent results — see `eigensolver_v3.ipynb` for the
working pipeline instead.

### What was tried

Attempted to extract the principal block sub-matrix corresponding to a
single surface (rather than solving on the full global flux Jacobian)
before running the eigensolver, with the goal of reducing problem size.

- **Test case:** 21,228-node cylinder mesh
- **Surfaces extracted:** flag 3 and flag 2 (left and right wall)
- **Method:** principal block submatrix (rows/cols restricted to surface
  node DOFs), as described in the surface-extraction utility function

### Result

Eigenvalues from the surface-restricted sub-matrix are **not consistent**
with eigenvalues obtained from the full global Jacobian.

### Why (likely cause)

The principal submatrix drops all coupling terms between surface nodes
and their interior neighbors (see: block terms $J_{si}$, $J_{is}$ are
discarded, not folded in). This changes the eigenproblem being solved —
it no longer represents the surface's role within the coupled
interior+surface system, so its spectrum has no reason to match the
full-domain spectrum. A Schur complement reduction would be needed
instead to correctly account for interior feedback, rather than a plain
submatrix extraction.

### Alternative approach (already validated elsewhere)

Adopt the **spanwise-homogeneity assumption** of Schmid et al. (2017) —
a different, already-tested dimensionality-reduction strategy for the
global stability problem, rather than naive surface-block extraction.

In [1]:
"""
Global flux Jacobian stability analysis pipeline.

1. Reads a sparse Jacobian saved in scipy's .npz (CSR) format.
2. Solves the eigenproblem with SLEPc shift-invert, sigma settable
   as a complex number, targeting eigenvalues near sigma (robust
   convergence for interior/near-marginal modes).
3. Reports the 10 eigenvalues with the largest growth rate among
   those found, with diagnostics to help you judge whether to
   trust each one.

Requires the complex-scalar PETSc/SLEPc build (petsc-complex env),
since sigma is a general complex number here.
"""
import numpy as np
import matplotlib.pyplot as plt
import scipy.sparse as sp
from petsc4py import PETSc
from slepc4py import SLEPc



# =====================================================================
# 1. Read the Jacobian
# =====================================================================
def read_jacobian(path):
    """Load a sparse Jacobian saved via scipy.sparse.save_npz.
    Returns a CSR matrix cast to PETSc's scalar type (complex128
    in this environment)."""
    J = sp.load_npz(path)
    J = J.tocsr().astype(PETSc.ScalarType)
    print(f"Loaded Jacobian: shape={J.shape}, nnz={J.nnz}, "
          f"density={J.nnz / (J.shape[0]*J.shape[1]):.2e}")
    return J


def scipy_csr_to_petsc(J_csr):
    Mat = PETSc.Mat().createAIJ(size=J_csr.shape,
                                  csr=(J_csr.indptr, J_csr.indices, J_csr.data))
    Mat.assemble()
    return Mat


# =====================================================================
# 2. Solve with shift-invert
# =====================================================================
def solve_shift_invert(J, sigma, nev=10, ncv=None, tol=1e-10, max_it=2000):
    """
    sigma  : complex shift -- place this near where you expect the
             largest-growth-rate / marginal eigenvalues to sit
             (e.g. 0+0j, or 0 + 1j*omega_guess if you have a frequency
             estimate from a prior run or physical intuition).
    nev    : number of eigenvalues requested.
    ncv    : Krylov subspace size. Larger = more robust convergence,
             especially for clustered eigenvalues near a Hopf crossing,
             at the cost of more memory/compute per iteration. A common
             rule of thumb is ncv >= 2*nev, with more headroom (3-4x)
             if you expect closely spaced eigenvalues. Default here
             picks max(3*nev, 30), capped at the matrix dimension.
    """
    n = J.getSize()[0]
    if ncv is None:
        ncv = min(n, max(3 * nev, 30))

    E = SLEPc.EPS().create()
    E.setOperators(J)
    E.setProblemType(SLEPc.EPS.ProblemType.NHEP)  # non-symmetric flux Jacobian
    E.setType(SLEPc.EPS.Type.KRYLOVSCHUR)
    E.setDimensions(nev=nev, ncv=ncv)
    E.setTolerances(tol=tol, max_it=max_it)

    st = E.getST()
    st.setType(SLEPc.ST.Type.SINVERT)
    st.setShift(sigma)

    ksp = st.getKSP()
    ksp.setType('preonly')
    pc = ksp.getPC()
    pc.setType('lu')
    try:
        pc.setFactorSolverType('mumps')
        solver_used = 'mumps'
    except PETSc.Error:
        pc.setFactorSolverType('petsc')
        solver_used = 'petsc (built-in, no mumps found)'

    E.setTarget(sigma)
    E.setWhichEigenpairs(SLEPc.EPS.Which.TARGET_MAGNITUDE)
    E.setFromOptions()

    print(f"Solving: sigma={sigma}, nev={nev}, ncv={ncv}, "
          f"factorization={solver_used}")
    E.solve()
    return E


# =====================================================================
# 3. Extract and report results
# =====================================================================
def report_results(E, J, nev, residual_tol=1e-6):
    nconv = E.getConverged()
    nev_requested = E.getDimensions()[0]

    print(f"\nConverged eigenpairs: {nconv} / {nev_requested} requested")
    if nconv < nev_requested:
        print("  WARNING: fewer eigenpairs converged than requested.")
        print("  Consider: increasing ncv, increasing max_it, or checking")
        print("  whether sigma is placed sensibly relative to the spectrum.")

    vr, vi = J.createVecs()
    results = []
    for i in range(nev):
        val = E.getEigenpair(i, vr, vi)
        err = E.computeError(i)
        results.append({
            'eigenvalue': val,
            'residual': err,
            'vec_real': vr.getArray().copy(),
            'vec_imag': vi.getArray().copy(),
        })

    # sort by largest growth rate (real part), descending
    # results.sort(key=lambda r: -r['eigenvalue'].real)
    top_results = results[:nev]

    print(f"\n{'#':>3} {'Re(lambda)':>14} {'Im(lambda)':>14} {'residual':>12}  status")
    print("-" * 66)
    for i, r in enumerate(top_results):
        lam = r['eigenvalue']
        err = r['residual']
        if err > residual_tol:
            status = "UNRELIABLE (residual above tol)"
        elif lam.real > 0:
            status = "UNSTABLE"
        elif abs(lam.real) < 1e-3:
            status = "MARGINAL (near Re=0 -- check carefully)"
        else:
            status = "stable"
        print(f"{i:>3} {lam.real:>14.6f} {lam.imag:>14.6f} {err:>12.2e}  {status}")

    return top_results


# =====================================================================
# Print ALL converged eigenvalues (not just top 10)
# =====================================================================
def print_eigenvalues(results, residual_tol=1e-6):
    print(f"\n{len(results)} eigenvalues:")
    print(f"{'#':>3} {'Re(lambda)':>14} {'Im(lambda)':>14} {'residual':>12}  status")
    print("-" * 66)
    for i, r in enumerate(results):
        lam = r['eigenvalue']
        err = r['residual']
        if err > residual_tol:
            status = "UNRELIABLE (residual above tol)"
        elif lam.real > 0:
            status = "UNSTABLE"
        elif abs(lam.real) < 1e-3:
            status = "MARGINAL (near Re=0)"
        else:
            status = "stable"
        print(f"{i:>3} {lam.real:>14.6f} {lam.imag:>14.6f} {err:>12.2e}  {status}")


# =====================================================================
# Plot ALL converged eigenvalues on the complex plane
# =====================================================================
def plot_eigenspectrum(results, residual_tol=1e-6, save_path='eigenspectrum.png'):
    eigs = np.array([r['eigenvalue'] for r in results])
    residuals = np.array([r['residual'] for r in results])
    reliable = residuals <= residual_tol

    fig, ax = plt.subplots(figsize=(7.5, 6))

    ax.axvspan(min(eigs.real.min(), -0.1) - 0.05, 0, color='tab:blue', alpha=0.06)
    ax.axvspan(0, max(eigs.real.max(), 0.1) + 0.05, color='tab:red', alpha=0.06)
    ax.axvline(0, color='black', lw=1.0)

    ax.scatter(eigs.real[reliable], eigs.imag[reliable],
               c='tab:blue', s=45, edgecolor='white', linewidth=0.5,
               label='converged (residual OK)', zorder=3)
    if (~reliable).any():
        ax.scatter(eigs.real[~reliable], eigs.imag[~reliable],
                   c='gray', s=45, marker='x',
                   label='residual above tolerance -- do not trust', zorder=3)

    # annotate the leading (largest growth rate) eigenvalue
    idx_lead = np.argmax(eigs.real)
    ax.annotate(f'lambda = {eigs[idx_lead].real:.4f} + {eigs[idx_lead].imag:.4f}j',
                xy=(eigs[idx_lead].real, eigs[idx_lead].imag),
                xytext=(10, 10), textcoords='offset points', fontsize=8)

    ax.set_xlabel('Re(lambda)  (growth rate)')
    ax.set_ylabel('Im(lambda)  (frequency, rad/s or non-dim)')
    ax.set_title('Eigenspectrum of the global flux Jacobian')
    ax.legend(fontsize=8, loc='best')
    ax.grid(alpha=0.2)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.show()
    print(f"\nSaved: {save_path}")

# =====================================================================
# Save eigenvalues + eigenvectors in a portable format (.npz)
# =====================================================================
def save_eigendata(E, J, sigma, out_path='eigendata.npz'):
    """
    Saves:
      eigenvalues : complex array, shape (nconv,)
      eigenvectors: complex array, shape (nconv, N) -- each row is one
                    eigenvector, full length N (matches Jacobian dimension)
      residuals   : real array, shape (nconv,)
      sigma       : the complex shift used for this solve
      nev, ncv    : solver settings used, for reproducibility
    """
    nconv = E.getConverged()
    N = J.getSize()[0]
    nev_requested, ncv_used, _ = E.getDimensions()

    vr, vi = J.createVecs()
    eigenvalues = np.zeros(nconv, dtype=complex)
    eigenvectors = np.zeros((nconv, N), dtype=complex)
    residuals = np.zeros(nconv)

    for i in range(nconv):
        val = E.getEigenpair(i, vr, vi)
        eigenvalues[i] = val
        residuals[i] = E.computeError(i)
        # PETSc splits real/imag into separate vecs even in a complex
        # build when using getEigenpair this way -- combine them here
        eigenvectors[i, :] = vr.getArray() + 1j * vi.getArray()

    np.savez(out_path,
             eigenvalues=eigenvalues,
             eigenvectors=eigenvectors,
             residuals=residuals,
             sigma=np.array([sigma]),
             nev=nev_requested,
             ncv=ncv_used)

    print(f"Saved {nconv} eigenpairs to {out_path}")
    print(f"  eigenvalues.shape  = {eigenvalues.shape}")
    print(f"  eigenvectors.shape = {eigenvectors.shape}  (row i = eigenvector for eigenvalues[i])")
    
# =====================================================================
# Interpretation guide (printed, not just code) -- read this after running
# =====================================================================
def print_interpretation_guide():
    print("""
--------------------------------------------------------------------
How to interpret this output:

1. Re(lambda) > 0  -> that mode grows in time -> globally unstable.
   Re(lambda) < 0  -> decays -> stable.
   Re(lambda) ~ 0  -> marginal; this is the Hopf-relevant regime.

2. A genuine Hopf bifurcation shows up as a COMPLEX-CONJUGATE PAIR
   (nonzero Im(lambda), and you should see its conjugate elsewhere
   in the list or in a re-run with wider nev) with Re(lambda) crossing
   zero as you vary your control parameter (Reynolds number). A real
   eigenvalue crossing zero alone would indicate a different
   (steady/pitchfork) bifurcation, not Hopf.

3. Trust ONLY eigenpairs with residual comfortably below your solver
   tolerance (1e-6 to 1e-8 is typical). An eigenvalue with residual
   above tolerance is not converged -- don't draw physical conclusions
   from it, especially near Re(lambda)=0 where you need real precision.

4. If nconv < nev requested: SLEPc could not converge everything you
   asked for within max_it iterations at this ncv. Don't assume the
   unconverged ones don't exist -- widen ncv/max_it and re-run before
   concluding anything about the missing eigenvalues.

5. Sanity checks before trusting a Hopf conclusion:
   - Increase ncv and re-run: do the top eigenvalues change more than
     your tolerance? If yes, ncv was too small.
   - Nudge sigma slightly and re-run: do the same eigenvalues reappear?
     If eigenvalues disappear/appear with small sigma changes, you may
     be missing modes near the edge of what shift-invert "saw".
   - If you have a mesh-refinement study available, confirm the
     leading eigenvalue's real part doesn't move significantly under
     refinement -- a Hopf point that moves with mesh resolution isn't
     trustworthy yet.
--------------------------------------------------------------------
""")

In [2]:
# read the jacobian
Mesh = 21228
Re   = 60

Mach = 0.2
# AoA = 35

gamma = 1.4
R_gas = 287.0          # confirm units match the solver

# define path
# data_dir = "/home/ahf25/git/flux_jacobian/data/flux_jacobian_assembly_v4/v3_mesh"

data_dir = "../../data/flux_jacobian_assembly_v4/v1_mesh"
# data_dir = "./data/flux_jacobian"
JACOBIAN_PATH = f"{data_dir}/jacobian_cylinder_{Mesh}_Re{Re}_M{Mach}_fd.npz"   # <-- set to your actual file

# JACOBIAN_PATH = f"{data_dir}/jacobian_OAT15_M0.73_A35_fd_harten0.05.npz"

J_csr = read_jacobian(JACOBIAN_PATH)

Loaded Jacobian: shape=(106140, 106140), nnz=5961200, density=5.29e-04


In [6]:
import numpy as np
from pyau3d.utils import PltFileUtils, GrpFileUtils, UnkFileUtils
import matplotlib.pyplot as plt
from matplotlib.tri import Triangulation
import pandas as pd
import scipy.sparse as sp


# 1. Read neccessary files
Mesh = 21228
Re   = 60
Mach = 0.2
gamma = 1.4
R_gas = 287.0          # confirm units match the solver
mesh_ver = 1
case_name = "cylinder"

# dir = f"C:/Users/User/Git/flux_jacobian/cases/cylinder_{Mesh}_Re{Re}_M{Mach}"
dir = f"/home/ahf25/AME_project_data/CFD_2d_cylinder_all/Steady/Ma{Mach}/v{mesh_ver}_mesh/2d_cylinder_{Mesh}_Re{Re}"
# dir = f"/home/ahf25/OAT15/OAT15_M0.73_A35"
pltfile = PltFileUtils(f"{dir}/{case_name}.plt")
rstfile = UnkFileUtils(f"{dir}/{case_name}.rst", extend=False)  # both rst and unk are fine

In [22]:
# extract entries of the J_csr that only correspond to the surface nodes

# extract the nodes of given surface flag
flag = 4 # flag = 3 for left wall
surface_nodes, ifac3, ifac4 = pltfile.extract_surface_real(flag=flag)

In [23]:
def extract_surface_block_submatrix(J_csr, surface_nodes, ndof=5, sort_nodes=True):
    """
    Extract the block sub-matrix of the global flux Jacobian corresponding
    to a given set of surface (0-based) node indices.

    Since the Jacobian is block-structured with 5x5 blocks per node
    (interleaved ordering: node i occupies DOF indices [5*i, 5*i+5)),
    this pulls out exactly the block rows/columns belonging to the
    surface nodes -- i.e. R_i/U_i coupling (diagonal blocks) and
    R_i/U_j coupling between two SURFACE nodes i, j that happen to be
    directly connected (off-diagonal blocks).

    NOTE: this is a plain principal submatrix, not a Schur-complement
    reduction -- coupling terms between a surface node and an INTERIOR
    neighbor are simply dropped, not condensed/folded in. If you need
    the true reduced dynamics on the surface (accounting for how the
    interior responds), that requires a Schur complement instead --
    let me know if that's actually what you're after.

    Parameters
    ----------
    J_csr         : scipy.sparse CSR matrix, shape (ndof*N, ndof*N)
    surface_nodes : array of 0-based node indices on the surface
    ndof          : DOFs per node (5 for compressible NS conservative vars)
    sort_nodes    : sort surface_nodes ascending before extraction --
                    keeps the submatrix's internal ordering consistent
                    and reproducible run to run (recommended)

    Returns
    -------
    J_surface : scipy.sparse CSR matrix, shape (ndof*n_surf, ndof*n_surf)
    dof_index : the global DOF indices used, in the order they appear
                in J_surface (needed if you want to map results back
                to physical node IDs later)
    node_order: surface_nodes in the order actually used (matches
                J_surface's block ordering, i.e. block k in J_surface
                corresponds to node_order[k])
    """
    surface_nodes = np.asarray(surface_nodes)
    if sort_nodes:
        node_order = np.sort(surface_nodes)
    else:
        node_order = surface_nodes

    # build the flat DOF index list: for each node, its ndof consecutive
    # global indices, concatenated in node_order
    dof_index = (node_order[:, None] * ndof + np.arange(ndof)[None, :]).ravel()

    J_csr = J_csr.tocsr()
    J_surface = J_csr[dof_index, :][:, dof_index]
    J_surface = J_surface.tocsr()

    n_surf = len(node_order)
    print(f"Extracted surface block sub-matrix:")
    print(f"  surface nodes : {n_surf}")
    print(f"  sub-matrix shape : {J_surface.shape}")
    print(f"  sub-matrix nnz   : {J_surface.nnz}")

    return J_surface, dof_index, node_order

In [24]:
J_surface, dof_index, node_order = extract_surface_block_submatrix(
    J_csr, surface_nodes, ndof=5, sort_nodes=True)

# save it for later use (eigensolve, inspection, etc.)
sp.save_npz("jacobian_cylinder_surface_flag7.npz", J_surface)
np.savez("jacobian_cylidner_surface_flag7_nodemap.npz",
         dof_index=dof_index, node_order=node_order)

Extracted surface block sub-matrix:
  surface nodes : 6705
  sub-matrix shape : (33525, 33525)
  sub-matrix nnz   : 1129499


In [25]:
A_surface = -J_surface#adding negative sign at the front to define the system jacobian matrix A

# convert CSR matrix to petsc compatible form + adding negative sign at the front to define the system jacobian matrix A
A_surface = scipy_csr_to_petsc(A_surface)

In [26]:
import time

# set the shift SIGMA
f = 9.505 # frequency in Hz
# f = 17.67 # frequency in Hz from St = 0.07 for OAT15

SIGMA = 0.0 + f * 2 * np.pi*1j                            # <-- set your shift here

nev = 10
ncv = 300
t0 = time.perf_counter()

E = solve_shift_invert(A_surface, sigma=SIGMA, nev=nev, ncv = ncv)
# save run time 
t = time.perf_counter() - t0
top_results = report_results(E,A_surface, nev) # reporting the top 10 eigenvalues that are close to SIGMA
print(f"Run time: {t:.2f}s")

Solving: sigma=59.72167634474197j, nev=10, ncv=300, factorization=mumps

Converged eigenpairs: 15 / 10 requested

  #     Re(lambda)     Im(lambda)     residual  status
------------------------------------------------------------------
  0     -14.815314       0.000000     5.54e-07  stable
  1     -20.082033       0.000000     6.47e-07  stable
  2     -26.301261       2.182280     1.44e-07  stable
  3     -22.287136      -0.000000     1.61e-07  stable
  4     -22.767143       0.000000     3.13e-07  stable
  5     -23.038951       0.000000     2.46e-07  stable
  6     -23.326994       0.000000     2.35e-07  stable
  7     -33.713883       4.912990     2.06e-07  stable
  8     -24.557615      -0.000000     1.11e-07  stable
  9     -24.708499       0.000000     8.97e-08  stable
Run time: 7.34s
